# 01 From Scratch — Train Loop Anatomy

**Status:** Complete

**Learning outcome:** Build a complete PyTorch training loop from scratch with data loading, optimiser, early stopping, model checkpointing, and train/val/test split — the template for all subsequent experiments.

## Why This Matters

Every ML experiment follows the same loop: load data → forward → loss → backward → update → evaluate. Getting this loop right — with proper validation, early stopping, and checkpointing — is the difference between reproducible science and wasted GPU hours. This notebook builds the canonical loop we'll reuse across all MNEMOSYNE experiments.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.datasets import load_digits
import tempfile, os

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cpu')
print(f"Device: {device}")
print("Libraries loaded.")

Device: cpu
Libraries loaded.


## Data Preparation — Train / Val / Test Split

In [2]:
# Load sklearn digits and create PyTorch datasets
digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)

dataset = TensorDataset(X, y)

# 60/20/20 split
n = len(dataset)
n_train, n_val = int(0.6 * n), int(0.2 * n)
n_test = n - n_train - n_val
train_ds, val_ds, test_ds = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

batch_size = 64
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size)
test_dl = DataLoader(test_ds, batch_size=batch_size)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
print(f"Batches per epoch: {len(train_dl)}")

Train: 1078, Val: 359, Test: 360
Batches per epoch: 17


---
**Understanding Train/Val/Test Split and Data Leakage**

**Plain language:** Imagine studying for an exam with a textbook (training set), practice quizzes (validation set), and the final exam (test set). You learn from the textbook, check your understanding with practice quizzes and adjust your study strategy, then take the final exam exactly once to see your real score. If you peek at the final exam while studying, your score no longer reflects what you actually know — that's data leakage.

**Building intuition:** The training set is used to fit model parameters via gradient descent. The validation set guides hyperparameter choices (learning rate, architecture, when to stop training) without touching the parameters directly. The test set is evaluated exactly once at the end to give an unbiased estimate of generalisation performance. Data leakage occurs when information from the validation or test set influences training — for example, normalising features using statistics computed on the full dataset before splitting, or selecting a model architecture based on test set performance. Even subtle leakage (e.g., time-series data split randomly instead of chronologically) inflates reported metrics.

**Formal statement:** Given dataset $\mathcal{D}$, we partition into disjoint subsets $\mathcal{D}_{\text{train}}, \mathcal{D}_{\text{val}}, \mathcal{D}_{\text{test}}$ such that $\mathcal{D}_{\text{train}} \cap \mathcal{D}_{\text{val}} \cap \mathcal{D}_{\text{test}} = \emptyset$ and $\mathcal{D}_{\text{train}} \cup \mathcal{D}_{\text{val}} \cup \mathcal{D}_{\text{test}} = \mathcal{D}$. The model $f_\theta$ is fit on $\mathcal{D}_{\text{train}}$, hyperparameters are selected to minimise $\mathcal{L}(\mathcal{D}_{\text{val}})$, and the final reported metric is $\mathcal{L}(\mathcal{D}_{\text{test}})$. Data leakage is any violation of the constraint that $f_\theta$'s training procedure is statistically independent of $\mathcal{D}_{\text{test}}$.

---

## Model Definition

In [3]:
class DigitMLP(nn.Module):
    def __init__(self, n_in=64, n_hidden=128, n_out=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_out),
        )

    def forward(self, x):
        return self.net(x)

model = DigitMLP().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model}")
print(f"Parameters: {n_params:,}")

Model: DigitMLP(
  (net): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)
Parameters: 26,122


## Training Loop with Early Stopping and Checkpointing

In [4]:
@torch.no_grad()
def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += criterion(logits, yb).item() * len(xb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(xb)
    return total_loss / total, correct / total

# Training configuration
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 100
patience = 10

# Tracking
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
epochs_no_improve = 0
ckpt_dir = tempfile.mkdtemp()
ckpt_path = os.path.join(ckpt_dir, 'best_model.pt')

for epoch in range(epochs):
    # --- Train ---
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    # --- Evaluate ---
    train_loss, train_acc = evaluate(model, train_dl, criterion)
    val_loss, val_acc = evaluate(model, val_dl, criterion)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # --- Checkpointing ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        epochs_no_improve += 1

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train loss: {train_loss:.4f} acc: {train_acc:.2%} | "
              f"Val loss: {val_loss:.4f} acc: {val_acc:.2%} | patience: {epochs_no_improve}/{patience}")

    # --- Early stopping ---
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)")
        break

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

Epoch   0 | Train loss: 2.0822 acc: 71.24% | Val loss: 2.1000 acc: 64.62% | patience: 0/10
Epoch  10 | Train loss: 0.1544 acc: 96.57% | Val loss: 0.1783 acc: 96.66% | patience: 0/10


Epoch  20 | Train loss: 0.0599 acc: 99.17% | Val loss: 0.1015 acc: 96.94% | patience: 0/10
Epoch  30 | Train loss: 0.0296 acc: 99.72% | Val loss: 0.0817 acc: 96.94% | patience: 0/10


Epoch  40 | Train loss: 0.0165 acc: 100.00% | Val loss: 0.0793 acc: 96.38% | patience: 7/10
Epoch  50 | Train loss: 0.0098 acc: 100.00% | Val loss: 0.0770 acc: 96.38% | patience: 7/10


Epoch  60 | Train loss: 0.0059 acc: 100.00% | Val loss: 0.0728 acc: 96.94% | patience: 6/10

Early stopping at epoch 64 (no improvement for 10 epochs)

Training complete. Best val loss: 0.0716


---
**Understanding The Six Steps of the Training Loop**

**Plain language:** Training a neural network is like adjusting a recipe after each taste test. First you clean your palate (zero_grad), then you cook the dish (forward pass), taste it and score how far off it is (compute loss), figure out which ingredients caused the problem (backward pass), make sure you don't over-correct any single ingredient (gradient clipping), and finally adjust your recipe (optimizer step). The order is crucial — if you don't clean your palate first, yesterday's taste still lingers and corrupts your judgement.

**Building intuition:** `optimizer.zero_grad()` resets all parameter gradients to zero; without this, PyTorch accumulates gradients across batches, which is rarely desired. The forward pass `logits = model(x)` computes predictions by passing data through each layer. `loss = criterion(logits, y)` measures prediction error as a scalar. `loss.backward()` runs backpropagation, computing $\partial\mathcal{L}/\partial\theta$ for every parameter $\theta$ via the chain rule. Optional gradient clipping (`torch.nn.utils.clip_grad_norm_`) caps the gradient magnitude to prevent exploding gradients. Finally, `optimizer.step()` updates parameters using the computed gradients (e.g., $\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$ for SGD). Reordering these steps — say, calling `step()` before `backward()` — would update weights using stale or zero gradients.

**Formal statement:** For each mini-batch $(X_b, y_b)$, the canonical training step is: (1) $\nabla_\theta \leftarrow 0$; (2) $\hat{y} = f_\theta(X_b)$; (3) $\mathcal{L} = L(\hat{y}, y_b)$; (4) $\nabla_\theta \mathcal{L} = \frac{\partial \mathcal{L}}{\partial \theta}$ via reverse-mode AD; (5) $\nabla_\theta \mathcal{L} \leftarrow \text{clip}(\nabla_\theta \mathcal{L}, \, c)$ where $\|\nabla_\theta \mathcal{L}\| \leq c$; (6) $\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$ (or the adaptive-rate variant). Step (1) must precede (4) to avoid gradient accumulation; step (4) must follow (3) because autodiff requires the computational graph built in (2)–(3).

---

---
**Understanding Early Stopping as Implicit Regularisation**

**Plain language:** Think of a student preparing a presentation. At first they learn the key ideas (good generalisation), but if they keep rehearsing too long they start memorising filler words and awkward pauses from their practice recording (overfitting to noise). Early stopping is like a mentor saying "you're ready — stop practising now" at exactly the right moment, before the memorisation kicks in.

**Building intuition:** During training, both training loss and validation loss decrease initially. At some point the model begins to memorise training-set noise: training loss keeps falling, but validation loss starts rising. Early stopping monitors validation loss and halts training after a fixed number of epochs (patience) with no improvement. This acts as a form of regularisation — it constrains model complexity not by penalising weights directly, but by limiting the number of optimisation steps, which limits how far parameters can drift from their initialisation. Models trained with early stopping tend to have smaller effective weight norms.

**Formal statement:** Let $\theta_t$ denote model parameters at epoch $t$, and let $t^* = \arg\min_{t} \mathcal{L}_{\text{val}}(\theta_t)$. Early stopping selects $\theta_{t^*}$ rather than $\theta_T$ where $T$ is the maximum epoch. For quadratic loss with learning rate $\eta$, gradient descent from small initialisation $\theta_0 \approx 0$ yields $\theta_t \approx (I - (I - \eta H)^t) H^{-1} X^\top y$, which mirrors the Tikhonov (L2) regularised solution $\hat{\theta}_\lambda = (H + \lambda I)^{-1} X^\top y$ with an implicit correspondence $\lambda \approx \frac{1}{\eta t}$. Fewer steps $\Leftrightarrow$ stronger regularisation.

---

## Load Best Checkpoint and Test

In [5]:
# Load best model and evaluate on held-out test set
model.load_state_dict(torch.load(ckpt_path, weights_only=True))
test_loss, test_acc = evaluate(model, test_dl, criterion)

print(f"Test set (from best checkpoint):")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.2%}")
assert test_acc > 0.90, f"Test accuracy {test_acc:.2%} too low"

# Clean up checkpoint
os.remove(ckpt_path)
os.rmdir(ckpt_dir)

Test set (from best checkpoint):
  Loss: 0.1561
  Accuracy: 95.83%


In [6]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves'); axes[0].legend()

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].axhline(test_acc, color='red', ls='--', label=f'Test ({test_acc:.2%})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curves'); axes[1].legend()

plt.tight_layout()
plt.savefig('../assets/train_loop_curves.png', dpi=100)
plt.show()
print("Training curves saved.")

Training curves saved.


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_57889/974159865.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# --- Enhanced Loss Curve with Overfitting Annotation ---
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

el_train_loss = history['train_loss']
el_val_loss = history['val_loss']
el_epochs_range = range(len(el_train_loss))

# Find best epoch (lowest val loss) — this is where early stopping saved the checkpoint
el_best_epoch = int(np.argmin(el_val_loss))

# Identify overfitting zone: epochs where val loss increases AND train loss decreases
el_overfit_start = None
for ei in range(el_best_epoch + 1, len(el_val_loss)):
    if el_val_loss[ei] > el_val_loss[el_best_epoch] and el_train_loss[ei] < el_train_loss[el_best_epoch]:
        el_overfit_start = ei
        break

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(el_epochs_range, el_train_loss, label='Train Loss', linewidth=2, color='#2196F3')
ax.plot(el_epochs_range, el_val_loss, label='Val Loss', linewidth=2, color='#FF5722')

# Vertical line at best epoch
ax.axvline(x=el_best_epoch, color='green', linestyle='--', linewidth=1.5,
           label=f'Best epoch (early stop checkpoint) = {el_best_epoch}')

# Shade the overfitting zone
if el_overfit_start is not None:
    ax.axvspan(el_overfit_start, len(el_train_loss) - 1, alpha=0.15, color='red',
               label='Overfitting zone')

# Annotate the gap between train and val loss at the last epoch
el_last = len(el_train_loss) - 1
el_gap = el_val_loss[el_last] - el_train_loss[el_last]
ax.annotate('',
            xy=(el_last, el_train_loss[el_last]),
            xytext=(el_last, el_val_loss[el_last]),
            arrowprops=dict(arrowstyle='<->', color='purple', lw=1.5))
ax.text(el_last + 0.5, (el_train_loss[el_last] + el_val_loss[el_last]) / 2,
        f'Gap: {el_gap:.4f}', fontsize=10, color='purple', va='center')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Enhanced Loss Curves with Overfitting Annotation', fontsize=14)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/train_loop_annotated.png', dpi=100)
plt.show()
print("Enhanced loss curve saved to ../assets/train_loop_annotated.png")

In [ ]:
# --- Gradient Norm Over Training ---
# Re-train a fresh model for 30 epochs, tracking mean gradient norm per epoch
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

gn_model = DigitMLP().to(device)
gn_criterion = nn.CrossEntropyLoss()
gn_optimizer = optim.Adam(gn_model.parameters(), lr=1e-3)
gn_epochs = 30
gn_grad_norms = []

for gn_epoch in range(gn_epochs):
    gn_model.train()
    gn_epoch_norms = []
    for gn_xb, gn_yb in train_dl:
        gn_xb, gn_yb = gn_xb.to(device), gn_yb.to(device)
        gn_optimizer.zero_grad()
        gn_logits = gn_model(gn_xb)
        gn_loss = gn_criterion(gn_logits, gn_yb)
        gn_loss.backward()
        # Compute total gradient norm before stepping
        gn_total_norm = 0.0
        for gn_p in gn_model.parameters():
            if gn_p.grad is not None:
                gn_total_norm += gn_p.grad.data.norm(2).item() ** 2
        gn_total_norm = gn_total_norm ** 0.5
        gn_epoch_norms.append(gn_total_norm)
        gn_optimizer.step()
    gn_grad_norms.append(np.mean(gn_epoch_norms))

# Plot gradient norm over training
gn_fig, gn_ax = plt.subplots(figsize=(8, 4))
gn_ax.plot(range(gn_epochs), gn_grad_norms, color='darkgreen', linewidth=2)
gn_ax.set_xlabel('Epoch', fontsize=12)
gn_ax.set_ylabel('Mean Gradient Norm (L2)', fontsize=12)
gn_ax.set_title('Gradient Norm Over Training', fontsize=14)
gn_ax.grid(True, alpha=0.3)

# Annotate first and last values
gn_ax.annotate(f'{gn_grad_norms[0]:.2f}', xy=(0, gn_grad_norms[0]),
               xytext=(3, gn_grad_norms[0] * 1.15),
               arrowprops=dict(arrowstyle='->', color='gray'),
               fontsize=10, color='darkgreen')
gn_ax.annotate(f'{gn_grad_norms[-1]:.4f}', xy=(gn_epochs - 1, gn_grad_norms[-1]),
               xytext=(gn_epochs - 8, max(gn_grad_norms) * 0.3),
               arrowprops=dict(arrowstyle='->', color='gray'),
               fontsize=10, color='darkgreen')

plt.tight_layout()
plt.savefig('../assets/gradient_norm_training.png', dpi=100)
plt.show()
print("Gradient norm plot saved to ../assets/gradient_norm_training.png")

## Structured Interpretation

1. **The canonical loop**: `model.train()` → `zero_grad()` → `forward` → `loss` → `backward()` → `step()` → `model.eval()` → `evaluate()`. Every PyTorch experiment follows this pattern.

2. **Early stopping** prevents overfitting by monitoring validation loss. The patience parameter controls how many epochs without improvement we tolerate before stopping. Too low patience risks stopping during a plateau; too high wastes compute.

3. **Checkpointing** saves the model state at the best validation loss, not the final epoch. This is critical because training loss can continue to decrease while validation loss increases (overfitting). Loading the best checkpoint for test evaluation gives the most honest performance estimate.

4. **Train/Val/Test split**: Train for fitting, validation for hyperparameter tuning and early stopping, test for final unbiased evaluation. Never use test set for any decision during training.

5. **`@torch.no_grad()`** on the evaluate function disables gradient computation, reducing memory usage and speeding up inference. Combined with `model.eval()` (which changes batchnorm/dropout behaviour), this ensures correct evaluation.